# 03 - Analysis

Three questions about the Bahrain 2022 race (dry):

1. **Pace:** who was fastest?
2. **Tyre strategy:** who stopped when, and on which compound?
3. **Tyre degradation:** how much slower does a tyre get with age?

The charts of these results are in `04_plots.ipynb`.

In [1]:
import os
import sys

# The notebooks live in notebooks/, the package in the project root.
sys.path.append(os.path.abspath(os.path.join("..")))

# Reload boxbox automatically when a .py file changes.
%load_ext autoreload
%autoreload 2

In [2]:
from boxbox.analysis import (
    FIRST_LAST_LAPS,
    MIN_CLEAN_LAPS,
    MIN_LAP_SHARE,
    driver_strategies,
    get_stints,
    race_pace,
    tyre_degradation,
)
from boxbox.cleaning import mark_clean_laps
from boxbox.data import get_laps, prepare_laps

In [3]:
laps = mark_clean_laps(prepare_laps(get_laps(2022, "bahrain")))
clean = laps[laps["is_clean"]]
print(len(clean), "clean laps of", len(laps))

854 clean laps of 1125


## Question 1: who had the best race pace?

**Definition:** the pace of a driver is the **median** time of their clean
laps. The best lap is not used (the question is about the whole race), and
the median is preferred to the mean because a few odd laps (traffic, a
mistake) do not move it.

First, the three candidates side by side.

In [4]:
summary = clean.groupby("drv")["time"].agg(["count", "median", "mean", "min"])
summary.sort_values("median")

,count,median,mean,min
drv,,,,
LEC,46,97.6930,97.599109,94.570
PER,42,98.0575,98.061405,96.089
VER,41,98.0910,97.847146,95.440
SAI,44,98.1505,98.017818,95.740
HAM,42,98.3805,98.303214,96.228
RUS,44,98.8215,98.626932,96.302
TSU,42,99.0400,99.354500,97.104
MAG,45,99.0580,99.092222,96.623
OCO,43,99.1880,99.289837,97.110


Median and mean look close, but they do not always give the same order.
These are the drivers whose rank changes with the choice:

In [5]:
summary["rank_median"] = summary["median"].rank().astype(int)
summary["rank_mean"] = summary["mean"].rank().astype(int)

changed = summary[summary["rank_median"] != summary["rank_mean"]]
changed.sort_values("rank_median")[["median", "mean", "rank_median", "rank_mean"]]

,median,mean,rank_median,rank_mean
drv,,,,
PER,98.0575,98.061405,2,4
VER,98.0910,97.847146,3,2
SAI,98.1505,98.017818,4,3
TSU,99.0400,99.354500,7,9
MAG,99.0580,99.092222,8,7
OCO,99.1880,99.289837,9,8
GAS,99.5130,99.527692,10,12
BOT,99.7390,99.406956,11,10
ALO,99.7850,99.537390,12,13


For example PER is ahead of VER by 0.03 s on the median, but VER is ahead of
PER by 0.21 s on the mean. The choice matters, and the median is the one
that is not pulled by a few odd laps.

### The function

`race_pace` returns one row per driver, from the fastest to the slowest: the
number of clean laps, the median lap time, and the **gap in seconds to the
fastest driver**. A driver needs at least `MIN_LAP_SHARE` (70%) of the clean
laps of the driver who has the most, so that a driver who retired early is
not compared with the others.

In [6]:
pace = race_pace(laps)
pace

,drv,team,clean_laps,median_lap_time,gap
0,LEC,Ferrari,46,97.6930,0.0000
1,PER,Red Bull Racing,42,98.0575,0.3645
2,VER,Red Bull Racing,41,98.0910,0.3980
3,SAI,Ferrari,44,98.1505,0.4575
4,HAM,Mercedes,42,98.3805,0.6875
5,RUS,Mercedes,44,98.8215,1.1285
6,TSU,AlphaTauri,42,99.0400,1.3470
7,MAG,Haas F1 Team,45,99.0580,1.3650
8,OCO,Alpine,43,99.1880,1.4950
9,GAS,AlphaTauri,39,99.5130,1.8200


In [7]:
threshold = pace["clean_laps"].max() * MIN_LAP_SHARE
print(f"minimum number of clean laps: {threshold:.1f}")
print(len(pace), "drivers in the table (nobody is excluded in this race)")

minimum number of clean laps: 32.2
20 drivers in the table (nobody is excluded in this race)


**Reading:** LEC is the fastest, PER and VER are about 0.4 s behind, and NOR
is last about 3.8 s behind.

**Limit:** the median mixes different tyres and different fuel loads (the car
gets lighter, the tyres wear). A driver who stopped early is not on the same
tyres as one who stayed out. This is a first-order measure of pace; the tyre
questions below compare laps on the same compound.

## Question 2: tyre strategy

A **stint** is the run between two pit stops. Strategy uses **all** the laps
(pit laps and safety-car laps are part of a stint), not only the clean ones.

First, two checks on the raw lap table that decided how a stint is described.

In [8]:
# 1. Is the compound the same for every lap of a stint?
print(laps.groupby(["drv", "stint"])["compound"].nunique().max())  # expect 1

# 2. Is a new stint always a new set of tyres? life_start 1 = a new set.
raw_stints = laps.groupby(["drv", "stint"]).agg(
    fresh=("fresh", "first"),
    life_start=("life", "min"),
)
print(raw_stints["fresh"].value_counts())
print((raw_stints["fresh"] == (raw_stints["life_start"] == 1)).all())

1
fresh
True     49
False    29
Name: count, dtype: int64
True


**Reading:** one compound per stint, so a stint can be described by one
compound. A new stint is **not** always a new set: 49 of the 78 stints start
on a new set (`life_start` 1) and 29 on a used one. `fresh` is True exactly
when `life_start` is 1, so it adds nothing and is not kept; `life_start`
says the same and also how worn the set was.

### The function `get_stints`

One row per driver and stint: compound, first and last lap, number of laps,
and the age of the set when it was fitted.

In [9]:
stints = get_stints(laps)
print(len(stints), "stints")
print(stints.groupby("drv")["stint"].max().describe())  # stops per driver + 1
stints.head(12)

78 stints
count    20.000000
mean      3.900000
std       0.307794
min       3.000000
25%       4.000000
50%       4.000000
75%       4.000000
max       4.000000
Name: stint, dtype: float64


,drv,stint,compound,start_lap,end_lap,laps,life_start
0,ALB,1,SOFT,1,13,13,1
1,ALB,2,MEDIUM,14,35,22,1
2,ALB,3,MEDIUM,36,46,11,3
3,ALB,4,SOFT,47,57,11,4
4,ALO,1,SOFT,1,11,11,4
5,ALO,2,MEDIUM,12,25,14,1
6,ALO,3,HARD,26,42,17,1
7,ALO,4,SOFT,43,57,15,1
8,BOT,1,SOFT,1,14,14,4
9,BOT,2,MEDIUM,15,36,22,1


Can a pit stop keep the same set of tyres? Then the age (`life`) would
continue instead of restarting. The next cell counts the stints that start on
the age where the previous stint of the same driver ended.

In [10]:
sets = laps.groupby(["drv", "stint"]).agg(
    life_start=("life", "min"),
    life_end=("life", "max"),
)
sets["previous_life_end"] = sets.groupby("drv")["life_end"].shift()
same_set = sets["life_start"] == sets["previous_life_end"] + 1
print(same_set.sum(), "stints start on the same set as the previous stint")

0 stints start on the same set as the previous stint


### The strategies

The strategy of a driver is the sequence of the compounds of his stints.

In [11]:
strategy = stints.groupby("drv")["compound"].agg("-".join)
strategy.value_counts()

compound
SOFT-SOFT-MEDIUM-SOFT      7
SOFT-MEDIUM-MEDIUM-SOFT    3
SOFT-MEDIUM-HARD-SOFT      2
SOFT-HARD-MEDIUM-SOFT      2
SOFT-MEDIUM-SOFT-SOFT      2
SOFT-MEDIUM-HARD           1
SOFT-MEDIUM-SOFT           1
MEDIUM-HARD-SOFT-SOFT      1
MEDIUM-SOFT-HARD-SOFT      1
Name: count, dtype: int64

### The function `driver_strategies`

One row per driver: strategy, stops, laps completed, whether he was
classified, position after lap 1, final position and positions gained.
The final position is built like the official classification: more laps
first, and the same number of laps ordered by the last position.

In [12]:
strategies = driver_strategies(laps)
strategies

,drv,team,strategy,stops,laps_completed,classified,pos_lap1,final_pos,positions_gained
0,LEC,Ferrari,SOFT-SOFT-MEDIUM-SOFT,3,57,True,1.0,1,0.0
1,SAI,Ferrari,SOFT-SOFT-MEDIUM-SOFT,3,57,True,3.0,2,1.0
2,HAM,Mercedes,SOFT-HARD-MEDIUM-SOFT,3,57,True,4.0,3,1.0
3,RUS,Mercedes,SOFT-HARD-MEDIUM-SOFT,3,57,True,7.0,4,3.0
4,MAG,Haas F1 Team,SOFT-SOFT-MEDIUM-SOFT,3,57,True,5.0,5,0.0
5,BOT,Alfa Romeo,SOFT-MEDIUM-MEDIUM-SOFT,3,57,True,14.0,6,8.0
6,OCO,Alpine,SOFT-MEDIUM-HARD-SOFT,3,57,True,10.0,7,3.0
7,TSU,AlphaTauri,SOFT-MEDIUM-SOFT-SOFT,3,57,True,12.0,8,4.0
8,ALO,Alpine,SOFT-MEDIUM-HARD-SOFT,3,57,True,9.0,9,0.0
9,ZHO,Alfa Romeo,SOFT-MEDIUM-MEDIUM-SOFT,3,57,True,19.0,10,9.0


**Reading:** 9 different compound sequences for 20 drivers; the most common is
`SOFT-SOFT-MEDIUM-SOFT` (7 drivers). 18 drivers started on SOFT and 19
finished on it, so the strategies differ mostly in the middle stints. 18
drivers made 3 stops; MSC made 2 and finished, GAS made 2 and retired (the
only driver not classified). The biggest gains since lap 1 are ZHO (+9) and
BOT (+8); PER (-12), VER (-17) and GAS (-12) lost places because they
retired, not because of their strategy.

**Limits:** `pos_lap1` is the position after lap 1, not the starting grid
(that would need another source). The label of a driver who retired is
incomplete. With 9 sequences for 20 drivers many have a single driver, so
this describes what happened: it does not prove which strategy was better.

## Question 3: tyre degradation

**Definition:** how many seconds a set of tyres loses per lap of age. For
each stint (clean laps only), the mean time of the first laps is compared
with the mean time of the last laps, and the difference is divided by the
difference in tyre age (`life`).

The age used is `life`, the real age of the set (a used set is already worn
when it is fitted). The two numbers that define the method are constants of
`boxbox.analysis`: the number of laps averaged at each end and the minimum
number of clean laps a stint needs.

In [13]:
print("laps averaged at each end:", FIRST_LAST_LAPS)
print("minimum clean laps per stint:", MIN_CLEAN_LAPS)

laps averaged at each end: 3
minimum clean laps per stint: 10


First, what the data looks like: the clean laps per compound, and one
stint (LEC, stint 1) as lap time against tyre age.

In [14]:
print(clean["compound"].value_counts(dropna=False))

one = clean[(clean["drv"] == "LEC") & (clean["stint"] == 1)]
one[["lap", "life", "time"]]

compound
SOFT      478
MEDIUM    287
HARD       89
Name: count, dtype: int64


,lap,life,time
214,2,2,97.853
215,3,3,98.272
216,4,4,98.414
217,5,5,98.471
218,6,6,98.712
219,7,7,98.835
220,8,8,98.951
221,9,9,98.807
222,10,10,99.123
223,11,11,99.306


**Reading:** SOFT 478, MEDIUM 287, HARD 89 clean laps (HARD has few). In
LEC's first stint the time goes from 97.853 s (life 2) to 99.629 s (life
14): about 0.15 s lost per lap of age. In later stints the times often fall
instead: the car gets lighter as the fuel burns, which makes it faster and
hides part of the wear. So the value measured is the **net** effect and
underestimates the real wear.

In [15]:
degradation = tyre_degradation(laps)
print(len(degradation), "stints kept of", len(stints))
degradation

48 stints kept of 78


,drv,stint,compound,clean_laps,degradation_per_lap
0,ALB,1,SOFT,11,0.260625
1,ALB,2,MEDIUM,20,0.065902
2,ALO,2,MEDIUM,11,0.041760
3,ALO,3,HARD,14,0.144091
4,BOT,1,SOFT,12,0.028259
5,BOT,2,MEDIUM,20,-0.001765
6,GAS,1,SOFT,12,0.125815
7,GAS,2,MEDIUM,16,0.099744
8,GAS,3,HARD,11,0.034167
9,HAM,2,HARD,13,0.052581


In [16]:
degradation.groupby("compound")["degradation_per_lap"].agg(
    ["count", "median", "min", "max"]
)

,count,median,min,max
compound,,,,
HARD,6,0.085778,0.034167,0.144091
MEDIUM,16,0.077436,-0.001765,0.171615
SOFT,26,0.125824,0.026933,0.300190


### Why at least 10 clean laps

The first version kept stints with 6 clean laps or more (68 stints). The
shortest stints gave extreme values, because with so few laps the two groups
are only about 3 laps of age apart, so 1 second of noise becomes about 0.3 s
per lap (ZHO stint 4: -0.40; ZHO stint 3: +0.31). Results of that run, by
stint length:

| Compound | Stints under 10 clean laps | Median | Range | Stints of 10 or more | Median |
|---|---|---|---|---|---|
| SOFT | 14 | 0.029 | -0.40 to 0.28 | 26 | 0.126 |
| MEDIUM | 5 | 0.133 | 0.05 to 0.31 | 16 | 0.077 |
| HARD | 1 | 0.067 | - | 6 | 0.086 |

Short stints pushed the SOFT median down and the MEDIUM median up, and there
were 6 negative values (1 with the final rule). So the minimum was raised to
10 clean laps. The cost: 30 of the 78 stints are left out, and HARD keeps 6.

### Reading

- **SOFT wears fastest:** median about 0.126 s per lap, against about 0.077
  (MEDIUM) and 0.086 (HARD).
- **MEDIUM and HARD cannot be ranked:** the medians are 0.008 s apart and
  HARD has only 6 stints.
- **The stints vary a lot:** SOFT goes from 0.03 to 0.30, so a single SOFT
  stint can wear less than a MEDIUM one.
- **One MEDIUM stint is just below zero** (BOT, stint 2, -0.0018): essentially
  no wear, within the noise.

**Limits:** the fuel effect is not removed (the values are the net effect);
traffic and track conditions are ignored; three laps at each end and the
minimum of 10 are choices; only Bahrain 2022 (dry) was checked; there is no
statistical test.

## Summary

1. **Pace:** LEC was fastest on median clean lap; PER and VER about 0.4 s
   behind; NOR last, 3.8 s behind.
2. **Strategy:** 9 compound sequences; almost everyone started and ended on
   SOFT and made 3 stops.
3. **Degradation:** SOFT loses about 0.13 s per lap of tyre age, MEDIUM and
   HARD about 0.08 to 0.09; MEDIUM and HARD cannot be ranked.